# Time Series Forecasting

*Generated by DoML — Do Machine Learning*

In [ ]:
# Reproducibility — REPR-01 (non-negotiable per CLAUDE.md)
import os, random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import json, warnings
from pathlib import Path
from datetime import date
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression, QuantileRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb
import pmdarima as pm
from prophet import Prophet
from IPython.display import Markdown, display
warnings.filterwarnings('ignore')

# Project root — REPR-02 (non-negotiable per CLAUDE.md)
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/home/jovyan/work'))

with open(PROJECT_ROOT / '.planning' / 'config.json') as f:
    config = json.load(f)

print(f"Project root: {PROJECT_ROOT}")
print(f"time_factor: {config.get('time_factor', False)}")

In [ ]:
# ── Runtime parameters (injected by /doml-forecasting workflow) ──────────────
# Do NOT edit these manually if invoked via /doml-forecasting.
# To run this notebook manually, set the values below.

HORIZON = None            # int: number of periods ahead to forecast (required)
TARGET_COL = None         # str: column name to forecast (required)
REGRESSORS = ""           # str: comma-separated regressor columns, or "" if none
SEASONALITY = "auto"      # str: "auto"|"daily"|"weekly"|"monthly"|"yearly"|"none"
INPUT_FILE_OVERRIDE = ""  # str: resolved path to data/processed/ input file

# Validate required parameters
if HORIZON is None:
    raise ValueError("HORIZON is required. Run: /doml-forecasting --horizon N --target COLUMN")
if TARGET_COL is None:
    raise ValueError("TARGET_COL is required. Run: /doml-forecasting --horizon N --target COLUMN")

HORIZON = int(HORIZON)
REGRESSORS_LIST = [r.strip() for r in REGRESSORS.split(',') if r.strip()] if REGRESSORS else []

print(f"Horizon: {HORIZON} periods")
print(f"Target:  {TARGET_COL}")
print(f"Regressors: {REGRESSORS_LIST or 'none'}")
print(f"Seasonality: {SEASONALITY}")

## 1. Data Load

Reads from `data/processed/` — the EDA and preprocessing steps are responsible for ensuring
the dataset has a parsed datetime index, is sorted chronologically, and has regular frequency.

Input file is resolved by the `/doml-forecasting` workflow from `config.json` `dataset.path`.
Override at invocation time with `--file` if needed.

In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'

if INPUT_FILE_OVERRIDE:
    input_path = Path(INPUT_FILE_OVERRIDE)
    if not input_path.is_absolute():
        input_path = PROJECT_ROOT / input_path
else:
    # Fallback: find first supported file in data/processed/
    SUPPORTED = {'.csv', '.parquet', '.xlsx'}
    files = sorted([f for f in processed_dir.glob('**/*') if f.is_file() and f.suffix.lower() in SUPPORTED])
    if not files:
        raise FileNotFoundError(
            f"No supported files found in {processed_dir}. "
            "Run /doml-data-understanding and preprocessing first."
        )
    input_path = files[0]

ext = input_path.suffix.lower()
if ext == '.csv':
    df_raw = pd.read_csv(input_path)
elif ext == '.parquet':
    df_raw = pd.read_parquet(input_path)
elif ext == '.xlsx':
    df_raw = pd.read_excel(input_path)
else:
    raise ValueError(f"Unsupported file format: {ext}")

# Identify datetime column (first datetime column or string-parseable column named 'date'/'time'/'timestamp')
dt_col = None
for col in df_raw.columns:
    if pd.api.types.is_datetime64_any_dtype(df_raw[col]):
        dt_col = col
        break
    if col.lower() in ('date', 'datetime', 'time', 'timestamp', 'period'):
        try:
            pd.to_datetime(df_raw[col])
            dt_col = col
            break
        except Exception:
            pass

if dt_col is None:
    raise ValueError(
        "No datetime column found. Ensure the processed dataset has a date/time column. "
        "Expected column name: 'date', 'datetime', 'timestamp', or 'time'."
    )

df_raw[dt_col] = pd.to_datetime(df_raw[dt_col])
df = df_raw.sort_values(dt_col).set_index(dt_col)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found. Available columns: {df.columns.tolist()}")

missing_regressors = [r for r in REGRESSORS_LIST if r not in df.columns]
if missing_regressors:
    raise ValueError(f"Regressor columns not found: {missing_regressors}. Available: {df.columns.tolist()}")

print(f"Input file:  {input_path.name}")
print(f"Date column: {dt_col}")
print(f"Shape:       {df.shape[0]:,} rows \u00d7 {df.shape[1]} columns")
print(f"Date range:  {df.index.min()} \u2192 {df.index.max()}")
print(f"Target stats: mean={df[TARGET_COL].mean():.3f}, std={df[TARGET_COL].std():.3f}, nulls={df[TARGET_COL].isna().sum()}")

## 2. Seasonality Detection

<!-- D-05: auto-detect via ACF/PACF; override with --seasonality flag -->
ACF/PACF analysis to detect dominant seasonal period. If `--seasonality` override was provided,
that value is used for all models instead of auto-detection. The detected (or overridden) period
is logged transparently — not applied silently.

In [ ]:
# Infer frequency from time index
if hasattr(df.index, 'freq') and df.index.freq is not None:
    inferred_freq = df.index.freq.name
else:
    df.index = pd.DatetimeIndex(df.index)
    inferred_freq = pd.infer_freq(df.index)

print(f"Inferred time index frequency: {inferred_freq}")

# Map frequency to candidate seasonal period
FREQ_TO_PERIOD = {
    'D': 7,    # daily data → weekly seasonality
    'W': 52,   # weekly data → yearly seasonality
    'M': 12,   # monthly → yearly
    'MS': 12,
    'Q': 4,    # quarterly → yearly
    'QS': 4,
    'H': 24,   # hourly → daily
    'T': 60,   # minute → hourly
    'B': 5,    # business day → weekly
}

if SEASONALITY != 'auto':
    PERIOD_MAP = {'daily': 1, 'weekly': 7, 'monthly': 30, 'yearly': 365, 'none': 1}
    M = PERIOD_MAP.get(SEASONALITY, 1)
    USE_SEASONAL = (SEASONALITY != 'none')
    print(f"Seasonality override: {SEASONALITY} → period m={M}")
else:
    # Use ACF to find the lag with highest autocorrelation beyond lag 1
    target_series = df[TARGET_COL].dropna()
    n_lags = min(len(target_series) // 2 - 1, 120)
    acf_vals = acf(target_series, nlags=n_lags, fft=True)

    # Candidate period from frequency map
    candidate_m = None
    for freq_prefix, period in FREQ_TO_PERIOD.items():
        if inferred_freq and inferred_freq.startswith(freq_prefix):
            candidate_m = period
            break

    # Find dominant ACF lag beyond lag 1
    if len(acf_vals) > 2:
        acf_beyond_1 = acf_vals[2:]
        dominant_lag = int(np.argmax(np.abs(acf_beyond_1)) + 2)
    else:
        dominant_lag = candidate_m or 1

    # Use candidate_m if it aligns with a strong ACF lag, else use dominant_lag
    M = candidate_m if candidate_m else dominant_lag
    USE_SEASONAL = (M > 1)
    print(f"ACF dominant lag: {dominant_lag}  |  Frequency-based candidate: {candidate_m}  |  Selected period m={M}")

    # ACF/PACF plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sm.graphics.tsa.plot_acf(target_series, lags=n_lags, ax=axes[0], title=f'ACF — {TARGET_COL}')
    sm.graphics.tsa.plot_pacf(target_series, lags=min(n_lags, len(target_series) // 2 - 1),
                               ax=axes[1], method='ywm', title=f'PACF — {TARGET_COL}')
    plt.suptitle(f'Seasonality Detection: detected period m={M}', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

display(Markdown(f"**Detected/applied seasonal period: m={M}** (USE_SEASONAL={USE_SEASONAL})"))

## 3. Lag Feature Engineering

<!-- D-06: global models (LightGBM, XGBoost, Linear) use lag features derived from target -->
Lag features and rolling statistics are derived from `TARGET_COL`. Lag windows are set
based on the detected seasonal period m. Regressor columns (if provided) are appended as
additional features for the global models.

In [ ]:
# Lag feature set derived from seasonal period m (D-06: Claude's discretion on exact lags)
# Lag windows: 1, 2, 3, m (seasonal lag), 2*m (double-seasonal)
LAG_WINDOWS = sorted(set([1, 2, 3, M, 2 * M]))

df_feat = df[[TARGET_COL] + REGRESSORS_LIST].copy()

for lag in LAG_WINDOWS:
    if lag < len(df_feat):
        df_feat[f'lag_{lag}'] = df_feat[TARGET_COL].shift(lag)

# Rolling mean and std at seasonal period
if M > 1 and M < len(df_feat):
    df_feat[f'rolling_mean_{M}'] = df_feat[TARGET_COL].shift(1).rolling(M).mean()
    df_feat[f'rolling_std_{M}'] = df_feat[TARGET_COL].shift(1).rolling(M).std()

# Drop rows with NaN from lag creation
df_feat = df_feat.dropna()

feature_cols = [c for c in df_feat.columns if c != TARGET_COL]
X_global = df_feat[feature_cols]
y_global = df_feat[TARGET_COL]

print(f"Feature matrix: {X_global.shape[0]:,} rows \u00d7 {X_global.shape[1]} features")
print(f"Lag columns: {[c for c in feature_cols if c.startswith('lag_')]}")
print(f"Rolling columns: {[c for c in feature_cols if c.startswith('rolling_')]}")
print(f"Regressor columns: {REGRESSORS_LIST or 'none'}")

## 4. TimeSeriesSplit Cross-Validation

<!-- FORE-03: TimeSeriesSplit ONLY — no random splits ever -->
All models are evaluated using `TimeSeriesSplit` with 5 folds. Each fold uses only
past data to predict future data — no data leakage. This is the **only** CV strategy
used in this notebook. `KFold`, `StratifiedKFold`, and `train_test_split` are not used
for model evaluation.

Metrics recorded per model per fold: MAE, RMSE, MAPE (skipped if target contains zeros).

In [ ]:
N_SPLITS = 5
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
leaderboard_rows = []

def mape(y_true, y_pred):
    """MAPE — returns None if any y_true is zero (avoids division by zero)."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    if np.any(y_true == 0):
        return None
    return float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)

def record_model(name, maes, rmses, mapes):
    mae_mean = float(np.mean(maes))
    rmse_mean = float(np.mean(rmses))
    mape_values = [m for m in mapes if m is not None]
    mape_mean = float(np.mean(mape_values)) if mape_values else None
    leaderboard_rows.append({
        'model': name,
        'cv_mae_mean': round(mae_mean, 4),
        'cv_rmse_mean': round(rmse_mean, 4),
        'cv_mape_mean': round(mape_mean, 4) if mape_mean is not None else None,
        'n_splits': N_SPLITS,
        'horizon': HORIZON,
        'target': TARGET_COL,
        'run_date': str(date.today()),
    })
    print(f"{name:30s}  MAE={mae_mean:.4f}  RMSE={rmse_mean:.4f}  MAPE={'N/A' if mape_mean is None else f'{mape_mean:.2f}%'}")

print(f"TimeSeriesSplit: {N_SPLITS} folds  (FORE-03 — no random splits)")

## 5. SeasonalNaive Baseline

<!-- MOD-07: baseline always in leaderboard -->
Repeats the last observed season's values. The simplest sensible baseline for seasonal data.
Any model that cannot beat SeasonalNaive provides no value over naive extrapolation.

**Prediction intervals:** Empirical — computed from residual standard deviation across folds.

In [ ]:
# SeasonalNaive: predict y[t] = y[t - m] for m = seasonal period (D-07)
target_series_full = df[TARGET_COL].dropna()
sn_maes, sn_rmses, sn_mapes = [], [], []
sn_residuals = []

for train_idx, test_idx in tscv.split(target_series_full):
    train_y = target_series_full.iloc[train_idx]
    test_y = target_series_full.iloc[test_idx]
    n_test = len(test_y)
    # Repeat last m values cyclically
    last_season = train_y.values[-M:] if len(train_y) >= M else train_y.values
    preds = np.tile(last_season, int(np.ceil(n_test / len(last_season))))[:n_test]
    residuals = test_y.values - preds
    sn_residuals.extend(residuals.tolist())
    sn_maes.append(mean_absolute_error(test_y, preds))
    sn_rmses.append(np.sqrt(mean_squared_error(test_y, preds)))
    sn_mapes.append(mape(test_y.values, preds))

record_model('SeasonalNaive', sn_maes, sn_rmses, sn_mapes)

# Forecast future HORIZON periods with 80% and 95% prediction intervals
residual_std = np.std(sn_residuals)
last_season_vals = target_series_full.values[-M:] if len(target_series_full) >= M else target_series_full.values
sn_forecast = np.tile(last_season_vals, int(np.ceil(HORIZON / len(last_season_vals))))[:HORIZON]
sn_pi80_lo = sn_forecast - 1.282 * residual_std   # 80% interval z=1.282
sn_pi80_hi = sn_forecast + 1.282 * residual_std
sn_pi95_lo = sn_forecast - 1.960 * residual_std   # 95% interval z=1.96
sn_pi95_hi = sn_forecast + 1.960 * residual_std

print(f"\nSeasonalNaive forecast for next {HORIZON} periods:")
print(f"  Point range: [{sn_forecast.min():.3f}, {sn_forecast.max():.3f}]")
print(f"  95% PI range: [{sn_pi95_lo.min():.3f}, {sn_pi95_hi.max():.3f}]")

## 6. ARIMA / SARIMA

<!-- MOD-04, D-04: univariate on TARGET_COL only via pmdarima.auto_arima -->
`pmdarima.auto_arima` selects optimal (p,d,q) order via stepwise search (AIC criterion).
If a seasonal period m > 1 was detected, SARIMA is used (`seasonal=True, m=M`).
Otherwise, plain ARIMA is used (`seasonal=False`).

**Prediction intervals:** Native confidence intervals from `get_forecast()` at alpha=0.2 (80%) and alpha=0.05 (95%).

In [ ]:
# ARIMA/SARIMA via pmdarima — univariate on TARGET_COL (D-04)
arima_series = df[TARGET_COL].dropna().reset_index(drop=True)
arima_maes, arima_rmses, arima_mapes = [], [], []

for train_idx, test_idx in tscv.split(arima_series):
    train_y = arima_series.iloc[train_idx].values
    test_y = arima_series.iloc[test_idx].values
    try:
        model = pm.auto_arima(
            train_y,
            seasonal=USE_SEASONAL,
            m=M if USE_SEASONAL else 1,
            stepwise=True,
            information_criterion='aic',
            suppress_warnings=True,
            error_action='ignore',
            random_state=SEED,
        )
        preds = model.predict(n_periods=len(test_y))
        arima_maes.append(mean_absolute_error(test_y, preds))
        arima_rmses.append(np.sqrt(mean_squared_error(test_y, preds)))
        arima_mapes.append(mape(test_y, preds))
    except Exception as e:
        print(f"ARIMA fold failed: {e} — skipping fold")

model_name_arima = 'SARIMA' if USE_SEASONAL else 'ARIMA'
record_model(model_name_arima, arima_maes, arima_rmses, arima_mapes)

# Fit final model on full series for forecasting
arima_final = pm.auto_arima(
    arima_series.values,
    seasonal=USE_SEASONAL,
    m=M if USE_SEASONAL else 1,
    stepwise=True,
    information_criterion='aic',
    suppress_warnings=True,
    error_action='ignore',
    random_state=SEED,
)
arima_forecast, arima_conf = arima_final.predict(n_periods=HORIZON, return_conf_int=True, alpha=0.05)
arima_pi95_lo, arima_pi95_hi = arima_conf[:, 0], arima_conf[:, 1]
_, arima_conf80 = arima_final.predict(n_periods=HORIZON, return_conf_int=True, alpha=0.20)
arima_pi80_lo, arima_pi80_hi = arima_conf80[:, 0], arima_conf80[:, 1]

print(f"\n{model_name_arima} order: {arima_final.order}  seasonal_order: {getattr(arima_final, 'seasonal_order', 'N/A')}")
print(f"Forecast {HORIZON} periods: [{arima_forecast.min():.3f}, {arima_forecast.max():.3f}]")

## 7. Prophet

<!-- MOD-04, D-04: uses add_regressor() for --regressors columns; univariate otherwise -->
Facebook Prophet is evaluated using TimeSeriesSplit. Future regressor values for the forecast
horizon are taken as the last known values (last-observation-carried-forward) — suitable for
planning regressors (e.g., planned promotions). Prophet uses its built-in uncertainty intervals.

In [ ]:
# Prophet — uses TimeSeriesSplit, no random splits (FORE-03)
# Prophet requires columns named 'ds' (date) and 'y' (target)
prophet_df = df[[TARGET_COL] + REGRESSORS_LIST].copy().reset_index()
prophet_df = prophet_df.rename(columns={prophet_df.columns[0]: 'ds', TARGET_COL: 'y'})

prophet_maes, prophet_rmses, prophet_mapes = [], [], []

split_indices = list(tscv.split(prophet_df))
for train_idx, test_idx in split_indices:
    train_pdf = prophet_df.iloc[train_idx]
    test_pdf = prophet_df.iloc[test_idx]
    try:
        m_prophet = Prophet(interval_width=0.95, yearly_seasonality='auto',
                            weekly_seasonality='auto', daily_seasonality='auto')
        for reg in REGRESSORS_LIST:
            m_prophet.add_regressor(reg)  # D-04: add_regressor() for each regressor
        m_prophet.fit(train_pdf)
        forecast_pdf = m_prophet.predict(test_pdf[['ds'] + REGRESSORS_LIST])
        preds = forecast_pdf['yhat'].values
        test_y = test_pdf['y'].values
        prophet_maes.append(mean_absolute_error(test_y, preds))
        prophet_rmses.append(np.sqrt(mean_squared_error(test_y, preds)))
        prophet_mapes.append(mape(test_y, preds))
    except Exception as e:
        print(f"Prophet fold failed: {e} — skipping fold")

record_model('Prophet', prophet_maes, prophet_rmses, prophet_mapes)

# Fit final Prophet model for HORIZON-step forecast
m_prophet_final = Prophet(interval_width=0.95, yearly_seasonality='auto',
                           weekly_seasonality='auto', daily_seasonality='auto')
for reg in REGRESSORS_LIST:
    m_prophet_final.add_regressor(reg)
m_prophet_final.fit(prophet_df)

future = m_prophet_final.make_future_dataframe(periods=HORIZON, freq=inferred_freq or 'D')
# Fill regressors in future periods with last known value (LOCF)
for reg in REGRESSORS_LIST:
    last_val = prophet_df[reg].iloc[-1]
    future[reg] = future[reg].fillna(last_val) if reg in future.columns else last_val
prophet_forecast_df = m_prophet_final.predict(future).tail(HORIZON)
prophet_forecast = prophet_forecast_df['yhat'].values
prophet_pi95_lo = prophet_forecast_df['yhat_lower'].values
prophet_pi95_hi = prophet_forecast_df['yhat_upper'].values
# 80% PI: refit with interval_width=0.80
m_p80 = Prophet(interval_width=0.80, yearly_seasonality='auto', weekly_seasonality='auto', daily_seasonality='auto')
for reg in REGRESSORS_LIST:
    m_p80.add_regressor(reg)
m_p80.fit(prophet_df)
future80 = m_p80.make_future_dataframe(periods=HORIZON, freq=inferred_freq or 'D')
for reg in REGRESSORS_LIST:
    future80[reg] = future80[reg].fillna(prophet_df[reg].iloc[-1]) if reg in future80.columns else prophet_df[reg].iloc[-1]
p80_forecast_df = m_p80.predict(future80).tail(HORIZON)
prophet_pi80_lo = p80_forecast_df['yhat_lower'].values
prophet_pi80_hi = p80_forecast_df['yhat_upper'].values
print(f"Prophet forecast {HORIZON} periods: [{prophet_forecast.min():.3f}, {prophet_forecast.max():.3f}]")

## 8. Global ML Models — LightGBM, XGBoost, Linear Regression

<!-- D-06: lag features + rolling stats; D-07: quantile regression for intervals -->
Evaluated on `TimeSeriesSplit` using the lag-feature matrix from Section 3.

**Prediction intervals:** Quantile regression — separate models fit at quantiles
0.10/0.90 (80% interval) and 0.025/0.975 (95% interval).

In [ ]:
def cv_global_model(name, model_fn):
    """Run TimeSeriesSplit CV for a global model factory function."""
    maes, rmses, mapes_list = [], [], []
    for train_idx, test_idx in tscv.split(X_global):
        X_tr, X_te = X_global.iloc[train_idx], X_global.iloc[test_idx]
        y_tr, y_te = y_global.iloc[train_idx], y_global.iloc[test_idx]
        m = model_fn()
        m.fit(X_tr, y_tr)
        preds = m.predict(X_te)
        maes.append(mean_absolute_error(y_te, preds))
        rmses.append(np.sqrt(mean_squared_error(y_te, preds)))
        mapes_list.append(mape(y_te.values, preds))
    record_model(name, maes, rmses, mapes_list)

def fit_quantile_intervals(model_q_fn, alpha_lo, alpha_hi):
    """Fit two quantile models for prediction interval bounds on full feature matrix."""
    m_lo = model_q_fn(alpha_lo)
    m_hi = model_q_fn(alpha_hi)
    m_lo.fit(X_global, y_global)
    m_hi.fit(X_global, y_global)
    return m_lo, m_hi

# Future feature matrix for HORIZON-step forecast (last known features repeated — simple extrapolation)
last_row = X_global.iloc[[-1]]
X_future = pd.concat([last_row] * HORIZON, ignore_index=True)

# ── LightGBM ──
cv_global_model('LightGBM', lambda: lgb.LGBMRegressor(n_estimators=100, random_state=SEED, verbose=-1))
lgb_final = lgb.LGBMRegressor(n_estimators=100, random_state=SEED, verbose=-1)
lgb_final.fit(X_global, y_global)
lgb_forecast = lgb_final.predict(X_future)
# Quantile intervals via LightGBM quantile objective
lgb_lo80 = lgb.LGBMRegressor(objective='quantile', alpha=0.10, n_estimators=100, random_state=SEED, verbose=-1)
lgb_hi80 = lgb.LGBMRegressor(objective='quantile', alpha=0.90, n_estimators=100, random_state=SEED, verbose=-1)
lgb_lo95 = lgb.LGBMRegressor(objective='quantile', alpha=0.025, n_estimators=100, random_state=SEED, verbose=-1)
lgb_hi95 = lgb.LGBMRegressor(objective='quantile', alpha=0.975, n_estimators=100, random_state=SEED, verbose=-1)
for qm in [lgb_lo80, lgb_hi80, lgb_lo95, lgb_hi95]:
    qm.fit(X_global, y_global)
lgb_pi80_lo = lgb_lo80.predict(X_future)
lgb_pi80_hi = lgb_hi80.predict(X_future)
lgb_pi95_lo = lgb_lo95.predict(X_future)
lgb_pi95_hi = lgb_hi95.predict(X_future)

# ── XGBoost ──
cv_global_model('XGBoost', lambda: xgb.XGBRegressor(n_estimators=100, random_state=SEED, verbosity=0))
xgb_final = xgb.XGBRegressor(n_estimators=100, random_state=SEED, verbosity=0)
xgb_final.fit(X_global, y_global)
xgb_forecast = xgb_final.predict(X_future)
# XGBoost quantile via empirical residual standard deviation
xgb_residuals = y_global.values - xgb_final.predict(X_global)
res_std = np.std(xgb_residuals)
xgb_pi80_lo = xgb_forecast - 1.282 * res_std
xgb_pi80_hi = xgb_forecast + 1.282 * res_std
xgb_pi95_lo = xgb_forecast - 1.960 * res_std
xgb_pi95_hi = xgb_forecast + 1.960 * res_std

# ── Linear Regression ──
cv_global_model('LinearRegression', lambda: LinearRegression())
lr_final = LinearRegression()
lr_final.fit(X_global, y_global)
lr_forecast = lr_final.predict(X_future)
# Linear quantile intervals via sklearn QuantileRegressor (D-07)
lr_lo80 = QuantileRegressor(quantile=0.10, alpha=0, solver='highs')
lr_hi80 = QuantileRegressor(quantile=0.90, alpha=0, solver='highs')
lr_lo95 = QuantileRegressor(quantile=0.025, alpha=0, solver='highs')
lr_hi95 = QuantileRegressor(quantile=0.975, alpha=0, solver='highs')
for qm in [lr_lo80, lr_hi80, lr_lo95, lr_hi95]:
    qm.fit(X_global, y_global)
lr_pi80_lo = lr_lo80.predict(X_future)
lr_pi80_hi = lr_hi80.predict(X_future)
lr_pi95_lo = lr_lo95.predict(X_future)
lr_pi95_hi = lr_hi95.predict(X_future)

print("\nAll 6 models evaluated. See leaderboard in next section.")

## 9. Forecast Leaderboard

<!-- FORE-01: leaderboard with all 6 models -->
All models ranked by CV MAE (lower is better). Metrics are averaged across all TimeSeriesSplit
folds — never training scores. The SeasonalNaive baseline is included so any model that
underperforms it is clearly visible.

In [ ]:
leaderboard = pd.DataFrame(leaderboard_rows).sort_values('cv_mae_mean').reset_index(drop=True)
display(Markdown("### CV Leaderboard (ranked by MAE, lower = better)"))
display(leaderboard[['model', 'cv_mae_mean', 'cv_rmse_mean', 'cv_mape_mean', 'n_splits', 'horizon', 'target']])

# Save leaderboard — separate from supervised leaderboard (D-06: models/forecast_leaderboard.csv)
models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(exist_ok=True)
lb_path = models_dir / 'forecast_leaderboard.csv'

if lb_path.exists():
    existing_lb = pd.read_csv(lb_path)
    leaderboard_all = pd.concat([existing_lb, leaderboard], ignore_index=True)
else:
    leaderboard_all = leaderboard.copy()

leaderboard_all.to_csv(lb_path, index=False)
print(f"\nLeaderboard saved: {lb_path.relative_to(PROJECT_ROOT)}")
print(f"Total rows in forecast_leaderboard.csv: {len(leaderboard_all)}")

best_model_name = leaderboard.iloc[0]['model']
best_mae = leaderboard.iloc[0]['cv_mae_mean']
sn_mae = leaderboard[leaderboard['model'] == 'SeasonalNaive']['cv_mae_mean'].values[0]
improvement_pct = (sn_mae - best_mae) / sn_mae * 100 if sn_mae > 0 else 0
print(f"\nBest model: {best_model_name}  (MAE={best_mae:.4f})")
print(f"vs SeasonalNaive baseline: {improvement_pct:.1f}% improvement")

In [ ]:
# Fan chart: best model forecast with 80% + 95% prediction intervals (FORE-02)
forecast_map = {
    'SeasonalNaive': (sn_forecast, sn_pi80_lo, sn_pi80_hi, sn_pi95_lo, sn_pi95_hi),
    'ARIMA': (arima_forecast, arima_pi80_lo, arima_pi80_hi, arima_pi95_lo, arima_pi95_hi),
    'SARIMA': (arima_forecast, arima_pi80_lo, arima_pi80_hi, arima_pi95_lo, arima_pi95_hi),
    'Prophet': (prophet_forecast, prophet_pi80_lo, prophet_pi80_hi, prophet_pi95_lo, prophet_pi95_hi),
    'LightGBM': (lgb_forecast, lgb_pi80_lo, lgb_pi80_hi, lgb_pi95_lo, lgb_pi95_hi),
    'XGBoost': (xgb_forecast, xgb_pi80_lo, xgb_pi80_hi, xgb_pi95_lo, xgb_pi95_hi),
    'LinearRegression': (lr_forecast, lr_pi80_lo, lr_pi80_hi, lr_pi95_lo, lr_pi95_hi),
}

best_key = next((k for k in forecast_map if k in best_model_name), None)
if best_key:
    fc, lo80, hi80, lo95, hi95 = forecast_map[best_key]
    hist_tail = df[TARGET_COL].dropna().tail(min(40, len(df)))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(range(len(hist_tail)), hist_tail.values, color='steelblue', label='Historical', linewidth=1.5)
    x_fc = range(len(hist_tail) - 1, len(hist_tail) + HORIZON - 1)
    ax.plot(x_fc, np.concatenate([[hist_tail.values[-1]], fc]), color='darkorange', linewidth=2, label=f'{best_key} forecast')
    ax.fill_between(x_fc,
                    np.concatenate([[hist_tail.values[-1]], lo95]),
                    np.concatenate([[hist_tail.values[-1]], hi95]),
                    alpha=0.15, color='darkorange', label='95% interval')
    ax.fill_between(x_fc,
                    np.concatenate([[hist_tail.values[-1]], lo80]),
                    np.concatenate([[hist_tail.values[-1]], hi80]),
                    alpha=0.25, color='darkorange', label='80% interval')
    ax.axvline(x=len(hist_tail) - 1, color='gray', linestyle='--', linewidth=1, label='Forecast start')
    ax.set_title(f'{best_key} — {HORIZON}-period Forecast with 80% and 95% Prediction Intervals')
    ax.set_xlabel('Time index')
    ax.set_ylabel(TARGET_COL)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f"Fan chart: {best_key} with 80% + 95% prediction intervals (FORE-02)")
else:
    print(f"No matching forecast array for best model '{best_model_name}' — plot skipped")

## 10. Forecast Actuals Tracking *(Placeholder — FORE-04 deferred to Milestone 4)*

<!-- D-01: FORE-04 deferred — multi-session state required -->
This section documents the **manual procedure** for comparing forecasts to actuals when
new data arrives. Automated tracking (`--compare` mode) is planned for Milestone 4.

### Manual comparison procedure

1. **Save this notebook's forecast output** — the `models/forecast_leaderboard.csv` contains
   point predictions and model metadata. To save raw forecast arrays, run:
   ```python
   import pandas as pd, numpy as np
   fc_df = pd.DataFrame({'period': range(1, HORIZON + 1), 'forecast': fc, 'pi80_lo': lo80, 'pi80_hi': hi80, 'pi95_lo': lo95, 'pi95_hi': hi95})
   fc_df.to_csv('models/forecast_values.csv', index=False)
   ```

2. **When actuals arrive** — load both files and compute:
   ```python
   actuals = pd.read_csv('data/processed/actuals.csv')  # your new data
   forecasts = pd.read_csv('models/forecast_values.csv')
   merged = forecasts.merge(actuals, on='period')
   mae = (merged['forecast'] - merged['actual']).abs().mean()
   coverage_80 = ((merged['actual'] >= merged['pi80_lo']) & (merged['actual'] <= merged['pi80_hi'])).mean()
   print(f"Post-hoc MAE: {mae:.4f}  |  80% interval coverage: {coverage_80:.1%}")
   ```

3. **Re-run `/doml-forecasting`** with additional periods of data in `data/processed/` to
   update the model and extend the horizon.

Automated FORE-04 (multi-session forecast tracking, `--compare` flag, drift detection)
is planned for Milestone 4.

## Caveats & Limitations

<!-- OUT-03: correlation is not causation disclaimer required in all stakeholder reports -->
- **Correlation is not causation.** Forecast models extrapolate historical patterns — they do
  not model the underlying causal mechanisms that drive the target variable.
- **TimeSeriesSplit CV** measures within-sample temporal generalization. Real-world performance
  may differ due to structural breaks, regime changes, or events not present in training data.
- **Lag features for global models** (LightGBM, XGBoost, Linear Regression) use last-observation-
  carried-forward for the forecast horizon — this is a simplification. Regressor future values
  are also LOCF unless you have actual future regressor values (e.g., planned promotions).
- **ARIMA/SARIMA** is univariate — it does not use regressor columns. For multivariate ARIMA,
  consider VAR models (deferred to Milestone 4).
- **Prophet regressors** in the forecast period use last-known values (LOCF). Replace these with
  actual planned values for a more accurate forecast.
- **Prediction intervals** for global models use quantile regression or empirical residual
  standard deviation — these are approximations, not exact Bayesian credible intervals.
- The `forecast_leaderboard.csv` accumulates across runs. Each run appends new rows —
  compare runs by `run_date` and `horizon` columns.